***Inspiration - https://www.kaggle.com/ritheshsreenivasan/clinical-text-classification***

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import string
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.manifold import TSNE

from nltk.tokenize import word_tokenize
from nltk.tokenize import sent_tokenize
from nltk.stem import WordNetLemmatizer 

from imblearn.over_sampling import SMOTE
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

In [ ]:
df = pd.read_csv(r"../input/medicaltranscriptions/mtsamples.csv")
df.head()

In [ ]:
df.columns

In [ ]:
df=df[['transcription','medical_specialty']]
df.head()

In [ ]:
df['medical_specialty'].value_counts()

Removing some categories as they have value_count less than 50

In [ ]:
counts = df['medical_specialty'].value_counts()

df = df[~df['medical_specialty'].isin(counts[counts < 50].index)]

In [ ]:
df['medical_specialty'].value_counts()

In [ ]:
plt.figure(figsize=(15,6))
plt.style.use(['dark_background'])
plt.xticks(rotation=90)
sns.countplot(x='medical_specialty', data = df )
plt.show()

In [ ]:
df.isna().sum()

In [ ]:
df.shape

In [ ]:
df.dropna(axis=0,inplace=True)

In [ ]:
df.shape

In [ ]:
print('Sample transcription 1:'+df.iloc[4]['transcription']+'\n')
print('Sample transcription 2:'+df.iloc[14]['transcription']+'\n')

In [ ]:
special_character_remover = re.compile('[/(){}\[\]\|@,;]')
extra_symbol_remover = re.compile('[^0-9a-z #+_]')
STOPWORDS = set(stopwords.words('english'))

In [ ]:
def clean_text(text):
    text = text.lower()
    text = special_character_remover.sub(' ',text)
    text = extra_symbol_remover.sub('',text)
    text = ' '.join(word for word in text.split() if word not in STOPWORDS)
    return text

def lemmatize_text(text):
    wordlist=[]
    lemmatizer = WordNetLemmatizer() 
    sentences=sent_tokenize(text)
    
    for sentence in sentences:
        words=word_tokenize(sentence)
        for word in words:
            wordlist.append(lemmatizer.lemmatize(word))    
    return ' '.join(wordlist) 

In [ ]:
nltk.download('punkt')
nltk.download('wordnet')

In [ ]:
df['transcription'] = df['transcription'].apply(clean_text)
df['transcription'] = df['transcription'].apply(lemmatize_text)

In [ ]:
print('Sample Transcription 1:'+df.iloc[5]['transcription']+'\n')
print('Sample Transcription 2:'+df.iloc[125]['transcription']+'\n')

In [ ]:
vectorizer = TfidfVectorizer(analyzer='word', stop_words='english',ngram_range=(1,3), max_df=0.75,min_df=5, use_idf=True, smooth_idf=True,sublinear_tf=True, max_features=1000)
tfIdfMat  = vectorizer.fit_transform(df['transcription'].tolist() )
feature_names = sorted(vectorizer.get_feature_names())
del feature_names[0:35]
print(feature_names)

In [ ]:
pca = PCA(n_components=0.95)
tfIdfMat_reduced = pca.fit_transform(tfIdfMat.toarray())
labels = df['medical_specialty'].tolist()
category_list = df.medical_specialty.unique()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(tfIdfMat_reduced, labels, stratify=labels,random_state=1)   

On Applying PCA, our number of features decreases from 1000 to 645

In [ ]:
print('Train_Set_Size:'+str(X_train.shape))
print('Test_Set_Size:'+str(X_test.shape))

In [ ]:
clf = LogisticRegression(penalty= 'elasticnet', solver= 'saga', l1_ratio=0.5, random_state=1).fit(X_train, y_train)
y_pred= clf.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix,classification_report

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels= category_list)

In [ ]:
print(classification_report(y_test,y_pred,labels=category_list))

In [ ]:
df['medical_specialty'].value_counts()

Surgery could belong to any of the categories, it is the superset of all classes like cardiology, orthopaedics. Similarly General Medicine, Pain Management all are supersets

In [ ]:
df = df[df['medical_specialty'] != ' Surgery']
df = df[df['medical_specialty'] != ' SOAP / Chart / Progress Notes']
df = df[df['medical_specialty'] != ' Emergency Room Reports']
df = df[df['medical_specialty'] != ' Discharge Summary']
df = df[df['medical_specialty'] != ' Office Notes']
df = df[df['medical_specialty'] != ' General Medicine']
df = df[df['medical_specialty'] != ' Pain Management']

In [ ]:
df['medical_specialty'].unique()

Similarly Neurosurgery is a branch of Neurology and Nephrology is a branch of Urology, so we combine them

In [ ]:
df.loc[df.medical_specialty == ' Neurosurgery', "medical_specialty"] = ' Neurology'
df.loc[df.medical_specialty == ' Nephrology', "medical_specialty"] = " Urology"

In [ ]:
df['medical_specialty'].value_counts()

In [ ]:
df.shape

In [ ]:
!pip install scispacy

- We will now use sciscpacy models to detect medical entities in our text 

- scispaCy is a Python package for processing biomedical, scientific or clinical text. 

In [ ]:
!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.4.0/en_ner_bionlp13cg_md-0.4.0.tar.gz

In [ ]:
import spacy
import en_ner_bionlp13cg_md

In [ ]:
nlp = en_ner_bionlp13cg_md.load()

In [ ]:
! pip install en_ner_bionlp13cg_md

In [ ]:
def process_Text( text):
    wordlist=[]
    doc = nlp(text)
    for ent in doc.ents:
        wordlist.append(ent.text)
    return ' '.join(wordlist)  

In [ ]:
df['transcription'] = df['transcription'].apply(process_Text)
df['transcription'] = df['transcription'].apply(lemmatize_text)
df['transcription'] = df['transcription'].apply(clean_text)

In [ ]:
vectorizer = TfidfVectorizer(analyzer='word', stop_words='english',ngram_range=(1,3), max_df=0.75,min_df=5, use_idf=True, smooth_idf=True,sublinear_tf=True, max_features=1000)
tfIdfMat  = vectorizer.fit_transform(df['transcription'].tolist() )
feature_names = sorted(vectorizer.get_feature_names())
print(feature_names)

In [ ]:
pca = PCA(n_components=0.95)
tfIdfMat_reduced = pca.fit_transform(tfIdfMat.toarray())
labels = df['medical_specialty'].tolist()
del feature_names[0:35]
category_list = df.medical_specialty.unique()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(tfIdfMat_reduced, labels, stratify=labels,random_state=1)   
print('Train_Set_Size:'+str(X_train.shape))
print('Test_Set_Size:'+str(X_test.shape))

In [ ]:
clf = LogisticRegression(penalty= 'elasticnet', solver= 'saga', l1_ratio=0.5, random_state=1).fit(X_train, y_train)
y_test_pred= clf.predict(X_test)

In [ ]:
print(classification_report(y_test,y_test_pred,labels=category_list))

In [ ]:
smote_over_sample = SMOTE(sampling_strategy='minority')
labels = df['medical_specialty'].tolist()
X, y = smote_over_sample.fit_resample(tfIdfMat_reduced, labels)

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y,random_state=1)   
print('Train_Set_Size:'+str(X_train.shape))
print('Test_Set_Size:'+str(X_test.shape))

In [ ]:
clf = LogisticRegression(penalty= 'elasticnet', solver= 'saga', l1_ratio=0.5, random_state=1).fit(X_train, y_train)
y_test_pred= clf.predict(X_test)

In [ ]:
print(classification_report(y_test,y_test_pred,labels=category_list))